# QUERY 4
Cuentas que cumplan con el patrón scatter-gather con una sola cuenta de separación,
para cuentas que hayan realizado transferencias en USD hacia 5 cuentas distintas dentro
del período [2022-09-01, 2022-09-05] 

In [22]:
import pandas as pd

# 1. Cargar datos
RUTA_DATASETS = "../../datasets"
NOMBRE_ARCHIVO_TRANSACCION = "q4_sample.csv"
NOMBRE_ARCHIVO_SOLUCION = "q4_solucion.csv"

# Definimos los dtypes básicos para evitar problemas de memoria o strings
transacciones = pd.read_csv(
    f"{RUTA_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCION}",
    dtype={
        "From Bank": "string", 
        "Account": "string",
        "To Bank": "string", 
        "Account.1": "string"
    }
)

# 2. Estandarizar nombres de columnas (si vienen con el formato original)
if "Account" in transacciones.columns and "Account.1" in transacciones.columns:
    transacciones = transacciones.rename(columns={
        "Account": "From Account", 
        "Account.1": "To Account"
    })

# 3. Formateo de fechas y montos
transacciones["Timestamp"] = pd.to_datetime(transacciones["Timestamp"])
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")

# 4. Filtro base: USD y sin nulos
df = transacciones[
    (transacciones["Payment Currency"] == "US Dollar") & 
    (transacciones["Amount Paid"].notna())
]

print(f"Transacciones válidas en USD: {len(df)}")

Transacciones válidas en USD: 18


In [23]:
# Filtramos por el período indicado asegurándonos de incluir el último día completo
df_periodo = df[
    (df["Timestamp"] >= "2022-09-01") & 
    (df["Timestamp"] <= "2022-09-05 23:59:59")
]

print(f"Transacciones en el período: {len(df_periodo)}")

Transacciones en el período: 18


In [24]:
# PASO 1: SCATTER (Dispersión)
# Nos quedamos con las relaciones únicas de A -> B
relaciones_ab = df_periodo[["From Account", "To Account"]].drop_duplicates()

# Contamos a cuántas cuentas destino (B) transfiere cada cuenta origen (A)
conteo_b_por_a = relaciones_ab.groupby("From Account")["To Account"].count()

# Filtramos las cuentas A que le transfirieron a EXACTAMENTE 5 cuentas distintas
# (Si necesitas que sean "al menos 5", cambia el == 5 por >= 5)
cuentas_a_validas = conteo_b_por_a[conteo_b_por_a == 5].index

# Obtenemos el listado de las transacciones A -> B filtradas
scatter_df = relaciones_ab[relaciones_ab["From Account"].isin(cuentas_a_validas)]


# PASO 2: GATHER (Recolección)
# Extraemos todas las relaciones B -> C dentro del mismo período
relaciones_bc = df_periodo[["From Account", "To Account"]].drop_duplicates()
relaciones_bc = relaciones_bc.rename(columns={
    "From Account": "Intermediate Account", 
    "To Account": "Final Account"
})

# Cruzamos los datos: Hacemos match donde la cuenta destino de A (B) sea la cuenta origen de C
patrones = pd.merge(
    scatter_df, 
    relaciones_bc, 
    left_on="To Account", 
    right_on="Intermediate Account",
    how="inner"
)

# Ahora agrupamos por Cuenta A y Cuenta C, y contamos por cuántas cuentas B pasaron
# Para que se cumpla el gather, las 5 cuentas B de un origen A deben converger en un mismo C
conteo_caminos = patrones.groupby(["From Account", "Final Account"])["Intermediate Account"].nunique().reset_index()

# Filtramos los casos donde los caminos (cuentas de separación) sean 5
resultado_scatter_gather = conteo_caminos[conteo_caminos["Intermediate Account"] == 5]

# Formateo final para presentar
resultado_scatter_gather = resultado_scatter_gather.rename(columns={
    "From Account": "From Account",
    "Final Account": "To Account",
    "Intermediate Account": "Amount Transactions"
}).reset_index(drop=True)

print(f"Se encontraron {len(resultado_scatter_gather)} patrones Scatter-Gather.")
# Guardamos el resultado en un CSV
resultado_scatter_gather.to_csv(f"{NOMBRE_ARCHIVO_SOLUCION}", index=False)
print(f"Resultado guardado en {RUTA_DATASETS}/{NOMBRE_ARCHIVO_SOLUCION}")

Se encontraron 1 patrones Scatter-Gather.
Resultado guardado en ../../datasets/q4_solucion.csv
